In [2]:
import os
import json
from groq import Groq
from dotenv import load_dotenv

In [3]:
load_dotenv()
api_key = os.getenv("GROQ_API_KEY")

client = Groq(api_key=api_key)

In [4]:
def ask_llm(prompt):

    completion = client.chat.completions.create(
        model="openai/gpt-oss-120b",
        messages=[{"role": "user", "content": prompt}],
        temperature=0.2,
        max_completion_tokens=1500
    )

    return completion.choices[0].message.content

In [5]:
def create_plan(task):

    prompt = f"""
You are an AI planning agent.

Break the task into steps.

Return ONLY JSON.

Example format:

{{
 "task": "{task}",
 "steps": [
   "step1",
   "step2",
   "step3"
 ]
}}

Task:
{task}
"""

    response = ask_llm(prompt)

    return json.loads(response)

In [6]:
def save_file(filename, content):

    with open(filename, "w") as f:
        f.write(content)

    return "File saved"


def read_file(filename):

    with open(filename, "r") as f:
        return f.read()


tools = {
    "save_file": save_file,
    "read_file": read_file
}


In [7]:
tool_description = """
Available tools:

save_file
input: filename, content

read_file
input: filename
"""

In [8]:
def decide_tool(step):

    prompt = f"""
You are an AI agent.

{tool_description}

Choose the best tool for the step.

Return ONLY JSON.

Example:

{{
 "tool": "tool_name",
 "input": {{
   "param": "value"
 }}
}}

Step:
{step}
"""

    response = ask_llm(prompt)

    return json.loads(response)

In [9]:
def run_agent(goal):

    memory = []

    for iteration in range(5):

        print("\nIteration:", iteration + 1)

        prompt = f"""
You are an autonomous AI agent.

Goal:
{goal}

Previous observations:
{memory}

Available tools:
save_file(filename, content)
read_file(filename)

Decide the next action.

Return ONLY JSON:

{{
 "tool": "tool_name",
 "input": {{ }}
}}
"""

        action = json.loads(ask_llm(prompt))

        tool_name = action["tool"]
        tool_input = action["input"]

        print("Action:", tool_name, tool_input)

        result = tools[tool_name](**tool_input)

        print("Observation:", result)

        memory.append(result)

In [10]:
run_agent("Create a file called notes.txt containing 'Learning Agentic AI'")


Iteration: 1
Action: save_file {'filename': 'notes.txt', 'content': 'Learning Agentic AI'}
Observation: File saved

Iteration: 2
Action: save_file {'filename': 'notes.txt', 'content': 'Learning Agentic AI'}
Observation: File saved

Iteration: 3
Action: save_file {'filename': 'notes.txt', 'content': 'Learning Agentic AI'}
Observation: File saved

Iteration: 4
Action: save_file {'filename': 'notes.txt', 'content': 'Learning Agentic AI'}
Observation: File saved

Iteration: 5
Action: save_file {'filename': 'notes.txt', 'content': 'Learning Agentic AI'}
Observation: File saved


In [11]:
# run_agent(plan)